# Logistic Regression
This Notebook utilises Logistic Regression to test Spark's Machine Learning library. While initial results appear promising, they are overly optimistic due to the skewed nature of the data. To address this, I quickly switched to Random Forests, which are better at dealing with imbalanced data classes.

In [18]:
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.ml.classification import LogisticRegression
#This transforms data
from pyspark.ml.feature import VectorAssembler


In [19]:
#Activate Spark
spark = SparkSession.builder.appName("LogisticRegression").getOrCreate()

In [20]:
df = spark.read.csv('data/diabetes_012.csv',inferSchema=True, header=True)

In [21]:
df.show(5)

+-----+------+--------+---------+----+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+----+---------+------+
|label|HighBP|HighChol|CholCheck| BMI|Smoker|Stroke|HeartDiseaseorAttack|PhysActivity|Fruits|Veggies|HvyAlcoholConsump|AnyHealthcare|NoDocbcCost|GenHlth|MentHlth|PhysHlth|DiffWalk|Sex| Age|Education|Income|
+-----+------+--------+---------+----+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+----+---------+------+
|  0.0|   1.0|     1.0|      1.0|40.0|   1.0|   0.0|                 0.0|         0.0|   0.0|    1.0|              0.0|          1.0|        0.0|    5.0|    18.0|    15.0|     1.0|0.0| 9.0|      4.0|   3.0|
|  0.0|   0.0|     0.0|      0.0|25.0|   1.0|   0.0|                 0.0|         1.0|   0.0|    0.0|              0.0|          0.0|        1.0|    3.0|     0.0|     0.0| 

In [22]:
inputColumns=["HighBP", "HighChol", "CholCheck","BMI","Smoker","Stroke","HeartDiseaseorAttack","PhysActivity","Fruits","Veggies","HvyAlcoholConsump","AnyHealthcare","NoDocbcCost","GenHlth","MentHlth","PhysHlth","DiffWalk","Sex", "Age","Education","Income"]

Feeding the data to a Vector Assembler converts all feature columns into a single column of vectors, essential for Spark to fit ML models on.

In [23]:
assembler = VectorAssembler(
    inputCols=inputColumns,
    outputCol="features")
vectorDF = assembler.transform(df)

In [24]:
vectorDF.show(5)

+-----+------+--------+---------+----+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+----+---------+------+--------------------+
|label|HighBP|HighChol|CholCheck| BMI|Smoker|Stroke|HeartDiseaseorAttack|PhysActivity|Fruits|Veggies|HvyAlcoholConsump|AnyHealthcare|NoDocbcCost|GenHlth|MentHlth|PhysHlth|DiffWalk|Sex| Age|Education|Income|            features|
+-----+------+--------+---------+----+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+----+---------+------+--------------------+
|  0.0|   1.0|     1.0|      1.0|40.0|   1.0|   0.0|                 0.0|         0.0|   0.0|    1.0|              0.0|          1.0|        0.0|    5.0|    18.0|    15.0|     1.0|0.0| 9.0|      4.0|   3.0|[1.0,1.0,1.0,40.0...|
|  0.0|   0.0|     0.0|      0.0|25.0|   1.0|   0.0|                 0.0|         1.0|  

Data is split into training and testing so as to avoid the Model from simply memorizing the test data. This avoids a key cause of overfitting

In [25]:
train_df, test_df = vectorDF.randomSplit(weights=[0.8, 0.2], seed=42)

lr = LogisticRegression(labelCol="label", featuresCol="features", maxIter=30, regParam=0.01)

lrmodel = lr.fit(train_df)

predictions = lrmodel.transform(test_df)

In [31]:
summary = lrmodel.summary
print('accuracy', summary.accuracy)
print('weighted precision', summary.weightedPrecision)
print('weighted recall', summary.weightedRecall)
print('training iterations', summary.totalIterations)

accuracy 0.8460076607225498
weighted precision 0.7985018698925549
weighted recall 0.8460076607225498
training iterations 30


In [32]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print('Test Accuracy on untrained data', accuracy)

Test Accuracy on untrained data 0.8486136929952933
